# LeetCode 3886 — Sum of Sortable Integers bằng Python

- Link: https://leetcode.com/problems/sum-of-sortable-integers/
- Độ khó: **Hard**
- Chủ đề: Ước số, xoay vòng (rotation), đếm "điểm giảm vòng", tiền tố max / hậu tố min, so khớp xâu (KMP) để kiểm tra xoay vòng

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def sortableIntegers(self, nums: list[int]) -> int:
```

## 1. Mục tiêu học

1. Nhận ra: một khối xoay vòng được thành dãy không giảm ⇔ khối có **tối đa một "điểm giảm vòng"**.
2. Tách điều kiện toàn cục thành: (a) mỗi khối tự sắp được; (b) tại mỗi **ranh giới khối**, max bên trái ≤ min bên phải.
3. Dùng **tiền tố max**, **hậu tố min**, **tiền tố số điểm giảm** để kiểm tra mỗi khối trong `O(1)` ⇒ mỗi ước `k` tốn `O(n/k)`.
4. Ước lượng `Σ_{k | n} n/k = σ(n)` (tổng ước) — rất nhỏ so với `n · d(n)`.
5. Biết kỹ thuật tổng quát "`B` có phải là một phép xoay của `A`" bằng **tìm `B` trong `A + A`** (KMP).

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng `nums` độ dài `n`. Số `k` gọi là **sắp được** nếu `k` chia hết `n` và ta có thể làm `nums` không giảm bằng cách:

1. chia `nums` thành các đoạn liên tiếp độ dài `k`;
2. **xoay vòng** từng đoạn (độc lập) bao nhiêu lần cũng được, sang trái hoặc phải.

Trả về **tổng** mọi `k` sắp được.

**Ví dụ 1:** `nums = [3,1,2]` → `3` (`k = 1` không được vì mảng chưa sắp; `k = 3`: xoay `[3,1,2]` thành `[1,2,3]`).

**Ví dụ 2:** `nums = [7,6,5]` → `0`.

**Ví dụ 3:** `nums = [5,8]` → `3` (đã sắp nên `k = 1, 2` đều được).

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `1 ≤ nums[i] ≤ 10^5`.

In [1]:
import random, time

examples = [([3, 1, 2], 3), ([7, 6, 5], 0), ([5, 8], 3)]
for nums, exp in examples:
    print(nums, "-> kỳ vọng", exp)

[3, 1, 2] -> kỳ vọng 3
[7, 6, 5] -> kỳ vọng 0
[5, 8] -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Liệt kê ước số trong `O(√n)`

Duyệt `i` từ 1 tới `√n`; nếu `i | n` thì cả `i` và `n / i` là ước (cẩn thận khi `i = n / i`).

In [2]:
def divisors(n):
    small, large = [], []
    i = 1
    while i * i <= n:
        if n % i == 0:
            small.append(i)
            if i != n // i:
                large.append(n // i)
        i += 1
    return small + large[::-1]

print(divisors(12), divisors(36), len(divisors(83160)))

[1, 2, 3, 4, 6, 12] [1, 2, 3, 4, 6, 9, 12, 18, 36] 128


### 3.2. Xoay vòng một list

Xoay trái `r` bước: `a[r:] + a[:r]`. Có `k` phép xoay khác nhau của một khối độ dài `k`.

In [3]:
a = [3, 1, 2]
print([a[r:] + a[:r] for r in range(len(a))])

[[3, 1, 2], [1, 2, 3], [2, 3, 1]]


### 3.3. Đếm điểm giảm vòng

Vị trí `j` là **điểm giảm vòng** nếu `a[j] > a[(j + 1) % k]`. Dãy không giảm khi đọc vòng thì chỉ có thể "gãy" ở **một** chỗ (chỗ nối cuối → đầu). Vậy khối xoay được thành không giảm ⇔ số điểm giảm vòng `≤ 1`.

In [4]:
def circular_drops(a):
    k = len(a)
    return sum(1 for j in range(k) if a[j] > a[(j + 1) % k])

for a in ([3, 1, 2], [1, 2, 3], [2, 2, 2], [2, 1, 3], [7, 6, 5]):
    print(a, "điểm giảm vòng:", circular_drops(a), "-> xoay được?", circular_drops(a) <= 1)

[3, 1, 2] điểm giảm vòng: 1 -> xoay được? True
[1, 2, 3] điểm giảm vòng: 1 -> xoay được? True
[2, 2, 2] điểm giảm vòng: 0 -> xoay được? True
[2, 1, 3] điểm giảm vòng: 2 -> xoay được? False
[7, 6, 5] điểm giảm vòng: 2 -> xoay được? False


### 3.4. Tiền tố max và hậu tố min

`pref_max[i] = max(nums[0..i-1])`, `suff_min[i] = min(nums[i..n-1])`. Tại vị trí cắt `i`, điều kiện "mọi phần tử bên trái ≤ mọi phần tử bên phải" là `pref_max[i] ≤ suff_min[i]`.

In [5]:
nums = [1, 3, 2, 4, 6, 5]
n = len(nums)
pm = [0] * (n + 1); sm = [float('inf')] * (n + 1)
for i in range(n): pm[i + 1] = max(pm[i], nums[i])
for i in range(n - 1, -1, -1): sm[i] = min(sm[i + 1], nums[i])
print("cắt tốt tại:", [i for i in range(n + 1) if pm[i] <= sm[i]])

cắt tốt tại: [0, 1, 3, 4, 6]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Xét `nums = [2, 1, 3, 5, 4, 6]` (`n = 6`), các ước `1, 2, 3, 6`:

- `k = 1`: không xoay được gì; mảng chưa sắp ⇒ ✘.
- `k = 2`: khối `[2,1]`, `[3,5]`, `[4,6]` ⇒ xoay `[1,2]`, `[3,5]`, `[4,6]` ⇒ `1,2,3,5,4,6` ✘ (ranh giới `5 > 4`).
- `k = 3`: khối `[2,1,3]` có 2 điểm giảm vòng (`2>1`, `3>2`) ⇒ không xoay được ⇒ ✘.
- `k = 6`: cả mảng một khối, điểm giảm vòng: `2>1`, `5>4`, `6>2` = 3 ⇒ ✘.

Đáp án `0`. Hai yếu tố quyết định: **từng khối** xoay được, và **ranh giới** giữa các khối.

In [6]:
nums = [2, 1, 3, 5, 4, 6]
for k in divisors(len(nums)):
    blocks = [nums[i:i + k] for i in range(0, len(nums), k)]
    print(f"k={k}: khối {blocks}, điểm giảm vòng {[circular_drops(b) for b in blocks]}")

k=1: khối [[2], [1], [3], [5], [4], [6]], điểm giảm vòng [0, 0, 0, 0, 0, 0]
k=2: khối [[2, 1], [3, 5], [4, 6]], điểm giảm vòng [1, 1, 1]
k=3: khối [[2, 1, 3], [5, 4, 6]], điểm giảm vòng [2, 2]
k=6: khối [[2, 1, 3, 5, 4, 6]], điểm giảm vòng [3]


## 5. Cách 1 — Brute Force: thử mọi phép xoay của mọi khối

Với mỗi ước `k`, với mỗi khối thử `k` phép xoay, chọn phép xoay cho khối không giảm (nếu có — mọi phép xoay hợp lệ cho cùng kết quả là `sorted(khối)`), rồi kiểm tra mảng ghép lại.

In [7]:
def is_sorted(a):
    return all(a[i] <= a[i + 1] for i in range(len(a) - 1))

def brute(nums):
    n = len(nums)
    ans = 0
    for k in divisors(n):
        out, ok = [], True
        for s in range(0, n, k):
            b = nums[s:s + k]
            good = next((b[r:] + b[:r] for r in range(k) if is_sorted(b[r:] + b[:r])), None)
            if good is None:
                ok = False
                break
            out += good
        if ok and is_sorted(out):
            ans += k
    return ans

for nums, exp in examples:
    print(brute(nums), "kỳ vọng", exp)

3 kỳ vọng 3
0 kỳ vọng 0
3 kỳ vọng 3


**Độ phức tạp:** mỗi khối `O(k²)` ⇒ mỗi ước `O(n · k)` ⇒ tổng `O(n · σ(n))` — với `n = 10^5` có thể tới `10^{10}`.

**Nút thắt:** thử từng phép xoay. Ta chỉ cần **biết có** phép xoay hợp lệ (điểm giảm vòng ≤ 1) và **giá trị min/max** của khối (không đổi khi xoay).

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** xoay không đổi **tập** phần tử của khối ⇒ sau khi xoay thành không giảm, khối bắt đầu bằng `min(khối)` và kết thúc bằng `max(khối)`.

**Insight 2:** mảng ghép không giảm ⇔ mỗi khối không giảm **và** tại mỗi ranh giới `s` (bội của `k`), `max(khối trước) ≤ min(khối sau)`. Kéo theo (do các khối đều đã sắp) điều kiện tương đương: `pref_max[s] ≤ suff_min[s]` tại mọi ranh giới.

**Insight 3 — O(1) mỗi khối:** số điểm giảm vòng của khối `[s, e]` = (số `j ∈ (s, e]` có `nums[j-1] > nums[j]`) + `[nums[e] > nums[s]]`. Phần đầu lấy từ tiền tố `pref_drop`.

**Insight 4 — tổng chi phí nhỏ:** mỗi ước `k` có `n/k` khối ⇒ tổng `Σ_{k|n} n/k = σ(n)` (tổng các ước của `n`), với `n ≤ 10^5` thì `σ(n) ≤ ~4·10^5`.

## 7. Các cách tiếp theo

- **Cách 2:** kiểm tra xoay vòng tổng quát bằng **KMP**: `sorted(khối)` có xuất hiện trong `khối + khối` không — `O(k log k)` mỗi khối (do sắp xếp).
- **Cách 3:** đếm điểm giảm vòng trực tiếp `O(k)` mỗi khối + so ranh giới — `O(n · d(n))`.
- **Cách 4:** tiền tố max / hậu tố min / tiền tố điểm giảm — `O(1)` mỗi khối, tổng `O(n + σ(n))` (lời giải nộp chính).

### 7.1. Cách 2 — "Có phải phép xoay?" bằng KMP

`B` là một phép xoay của `A` (cùng độ dài) ⇔ `B` là đoạn con của `A + A`. Áp dụng với `B = sorted(khối)`.

In [8]:
def kmp_find(text, pat):
    m = len(pat)
    fail = [0] * m
    j = 0
    for i in range(1, m):
        while j and pat[i] != pat[j]:
            j = fail[j - 1]
        if pat[i] == pat[j]:
            j += 1
        fail[i] = j
    j = 0
    for x in text:
        while j and x != pat[j]:
            j = fail[j - 1]
        if x == pat[j]:
            j += 1
        if j == m:
            return True
    return False

def kmp_solution(nums):
    n = len(nums)
    ans = 0
    for k in divisors(n):
        prev_max = float('-inf')
        ok = True
        for s in range(0, n, k):
            b = nums[s:s + k]
            sb = sorted(b)
            if not kmp_find(b + b, sb) or sb[0] < prev_max:
                ok = False
                break
            prev_max = sb[-1]
        if ok:
            ans += k
    return ans

for nums, exp in examples:
    print(kmp_solution(nums), "kỳ vọng", exp)

3 kỳ vọng 3
0 kỳ vọng 0
3 kỳ vọng 3


**Độ phức tạp Cách 2:** `O(n log k)` mỗi ước ⇒ `O(n log n · d(n))`. Ưu điểm: dùng được cho mọi "đích" (không chỉ sắp xếp).

### 7.2. Cách 3 — Đếm điểm giảm vòng trực tiếp

In [9]:
def direct_scan(nums):
    n = len(nums)
    ans = 0
    for k in divisors(n):
        ok = True
        prev_max = float('-inf')
        for s in range(0, n, k):
            e = s + k - 1
            drops = 0
            lo, hi = nums[s], nums[s]
            for j in range(s + 1, e + 1):
                if nums[j - 1] > nums[j]:
                    drops += 1
                lo = min(lo, nums[j]); hi = max(hi, nums[j])
            if nums[e] > nums[s]:
                drops += 1
            if drops > 1 or lo < prev_max:
                ok = False
                break
            prev_max = hi
        if ok:
            ans += k
    return ans

for nums, exp in examples:
    print(direct_scan(nums), "kỳ vọng", exp)

3 kỳ vọng 3
0 kỳ vọng 0
3 kỳ vọng 3


In [10]:
class SolutionScan:
    def sortableIntegers(self, nums: list[int]) -> int:
        return direct_scan(nums)

print([SolutionScan().sortableIntegers(a) for a, _ in examples])

[3, 0, 3]


**Độ phức tạp Cách 3:** `O(n)` mỗi ước ⇒ `O(n · d(n))`, với `d(n) ≤ 128` (cho `n ≤ 10^5`) ⇒ khoảng `10^7` bước.

### 7.3. Cách 4 — Tiền tố max / hậu tố min / tiền tố điểm giảm (lời giải chính)

**Giải thích từng đoạn:**

1. `pref_max[i]` = max của `nums[0..i-1]` (`pref_max[0] = 0`, vì `nums[i] ≥ 1`).
2. `suff_min[i]` = min của `nums[i..n-1]`.
3. `pref_drop[i]` = số `j ∈ [1, i]` với `nums[j-1] > nums[j]`.
4. Với mỗi ước `k`, mỗi khối bắt đầu ở `s`, kết thúc ở `e = s + k - 1`: `drops = pref_drop[e] - pref_drop[s] + [nums[e] > nums[s]]`. Khối hỏng nếu `drops > 1` **hoặc** `pref_max[s] > suff_min[s]`.

In [11]:
def prefix_solution(nums):
    n = len(nums)
    pref_max = [0] * (n + 1)
    for i, x in enumerate(nums):
        pref_max[i + 1] = max(pref_max[i], x)
    suff_min = [float('inf')] * (n + 1)
    for i in range(n - 1, -1, -1):
        suff_min[i] = min(nums[i], suff_min[i + 1])
    pref_drop = [0] * n
    for i in range(1, n):
        pref_drop[i] = pref_drop[i - 1] + (nums[i - 1] > nums[i])

    def ok(k):
        for s in range(0, n, k):
            e = s + k - 1
            drops = pref_drop[e] - pref_drop[s] + (nums[e] > nums[s])
            if drops > 1 or pref_max[s] > suff_min[s]:
                return False
        return True

    return sum(k for k in divisors(n) if ok(k))

for nums, exp in examples:
    print(prefix_solution(nums), "kỳ vọng", exp)

3 kỳ vọng 3
0 kỳ vọng 0
3 kỳ vọng 3


**Độ phức tạp Cách 4:** tiền xử lý `O(n)`; kiểm tra mọi ước `Σ n/k = σ(n)`; liệt kê ước `O(√n)` ⇒ tổng `O(n + σ(n)) = O(n log log n)`.

## 8. TRACE từng bước

Trace Cách 4 trên `nums = [2, 1, 4, 3, 6, 5]` (tự dựng).

| i | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|---|
| pref_max | 0 | 2 | 2 | 4 | 4 | 6 | 6 |
| suff_min | 1 | 1 | 3 | 3 | 5 | 5 | ∞ |
| pref_drop | 0 | 1 | 1 | 2 | 2 | 3 | – |

`k = 2`: khối `[2,1]` (s=0): drops = `pref_drop[1] - pref_drop[0]` + `[1 > 2]` = 1 ✔, ranh giới s=0: `0 ≤ 1` ✔; khối `[4,3]` (s=2): drops = 1 ✔, `pref_max[2] = 2 ≤ suff_min[2] = 3` ✔; khối `[6,5]` (s=4): ✔, `4 ≤ 5` ✔ ⇒ `k = 2` sắp được.

In [12]:
def trace(nums):
    n = len(nums)
    pref_max = [0] * (n + 1); suff_min = [float('inf')] * (n + 1); pref_drop = [0] * n
    for i, x in enumerate(nums): pref_max[i + 1] = max(pref_max[i], x)
    for i in range(n - 1, -1, -1): suff_min[i] = min(nums[i], suff_min[i + 1])
    for i in range(1, n): pref_drop[i] = pref_drop[i - 1] + (nums[i - 1] > nums[i])
    print("pref_max =", pref_max); print("suff_min =", suff_min); print("pref_drop =", pref_drop)
    total = 0
    for k in divisors(n):
        verdict = True
        for s in range(0, n, k):
            e = s + k - 1
            drops = pref_drop[e] - pref_drop[s] + (nums[e] > nums[s])
            good = drops <= 1 and pref_max[s] <= suff_min[s]
            print(f"  k={k} khối [{s},{e}]={nums[s:e + 1]} drops={drops} pref_max[s]={pref_max[s]} suff_min[s]={suff_min[s]} -> {'✔' if good else '✘'}")
            if not good:
                verdict = False; break
        total += k if verdict else 0
        print(f"k={k}: {'sắp được' if verdict else 'không'}")
    print("tổng =", total)

trace([2, 1, 4, 3, 6, 5])

pref_max = [0, 2, 2, 4, 4, 6, 6]
suff_min = [1, 1, 3, 3, 5, 5, inf]
pref_drop = [0, 1, 1, 2, 2, 3]
  k=1 khối [0,0]=[2] drops=0 pref_max[s]=0 suff_min[s]=1 -> ✔
  k=1 khối [1,1]=[1] drops=0 pref_max[s]=2 suff_min[s]=1 -> ✘
k=1: không
  k=2 khối [0,1]=[2, 1] drops=1 pref_max[s]=0 suff_min[s]=1 -> ✔
  k=2 khối [2,3]=[4, 3] drops=1 pref_max[s]=2 suff_min[s]=3 -> ✔
  k=2 khối [4,5]=[6, 5] drops=1 pref_max[s]=4 suff_min[s]=5 -> ✔
k=2: sắp được
  k=3 khối [0,2]=[2, 1, 4] drops=2 pref_max[s]=0 suff_min[s]=1 -> ✘
k=3: không
  k=6 khối [0,5]=[2, 1, 4, 3, 6, 5] drops=4 pref_max[s]=0 suff_min[s]=1 -> ✘
k=6: không
tổng = 2


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def sortableIntegers(self, nums: list[int]) -> int:
        n = len(nums)
        pref_max = [0] * (n + 1)
        for i, x in enumerate(nums):
            pref_max[i + 1] = pref_max[i] if pref_max[i] > x else x
        suff_min = [10**9] * (n + 1)
        for i in range(n - 1, -1, -1):
            suff_min[i] = nums[i] if nums[i] < suff_min[i + 1] else suff_min[i + 1]
        pref_drop = [0] * n
        for i in range(1, n):
            pref_drop[i] = pref_drop[i - 1] + (nums[i - 1] > nums[i])

        def ok(k):
            for s in range(0, n, k):
                e = s + k - 1
                drops = pref_drop[e] - pref_drop[s] + (nums[e] > nums[s])
                if drops > 1 or pref_max[s] > suff_min[s]:
                    return False
            return True

        ans = 0
        i = 1
        while i * i <= n:
            if n % i == 0:
                if ok(i):
                    ans += i
                j = n // i
                if j != i and ok(j):
                    ans += j
            i += 1
        return ans

sol = Solution()
for nums, exp in examples:
    got = sol.sortableIntegers(nums)
    print(nums, "->", got, "✔" if got == exp else "✘")

[3, 1, 2] -> 3 ✔
[7, 6, 5] -> 0 ✔
[5, 8] -> 3 ✔


## 10. Vì sao thuật toán đúng

1. **Khối xoay được ⇔ ≤ 1 điểm giảm vòng.** (⇒) Nếu một phép xoay cho dãy không giảm `b₀ ≤ … ≤ b_{k-1}`, đọc vòng chỉ có thể giảm ở cặp `(b_{k-1}, b₀)`. Phép xoay không đổi tập các cặp kề nhau vòng, nên khối gốc có ≤ 1 điểm giảm vòng. (⇐) Nếu không có điểm giảm vòng thì mọi phần tử bằng nhau. Nếu có đúng một tại `j`, xoay để bắt đầu ở `j + 1`: mọi cặp kề nhau (không vòng) đều không giảm.
2. **Kết quả xoay là duy nhất:** mọi phép xoay hợp lệ đều cho `sorted(khối)`.
3. **Ranh giới:** mảng ghép từ các khối đã sắp là không giảm ⇔ tại mỗi ranh giới, phần tử cuối khối trước (= max khối trước) ≤ phần tử đầu khối sau (= min khối sau). Khi mọi khối đã sắp, điều này tương đương `max(nums[0..s-1]) ≤ min(nums[s..n-1])` tại mọi ranh giới `s` (bắc cầu qua các khối).
4. **Công thức điểm giảm:** điểm giảm bên trong khối là các `j ∈ (s, e]` với `nums[j-1] > nums[j]` (đếm bằng hiệu tiền tố), cộng cặp vòng `(e, s)`. Khi `k = 1`: `pref_drop[s] - pref_drop[s] + (nums[s] > nums[s]) = 0` ✔.

## 11. Phân tích Time / Space Complexity

Đặt `n = len(nums)`, `d(n)` = số ước, `σ(n)` = tổng ước.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi phép xoay | `O(n · σ(n))` | `O(n)` |
| 2. KMP trên `khối + khối` | `O(n log n · d(n))` | `O(n)` |
| 3. Đếm điểm giảm trực tiếp | `O(n · d(n))` | `O(1)` thêm |
| 4. Tiền tố | `O(n + σ(n))` | `O(n)` |

Ví dụ `n = 83160` có `d(n) = 128`, `σ(n) ≈ 3,7·10^5`.

In [14]:
n = 83160
ds = divisors(n)
print("d(n) =", len(ds), "| σ(n) =", sum(ds), "| n·d(n) =", n * len(ds))

d(n) = 128 | σ(n) = 345600 | n·d(n) = 10644480


## 12. Bộ test

In [15]:
def run_all(nums, with_brute=True):
    res = [kmp_solution(nums), direct_scan(nums), prefix_solution(nums), Solution().sortableIntegers(nums)]
    if with_brute:
        res.append(brute(nums))
    return res

for nums, exp in examples:
    assert all(v == exp for v in run_all(nums))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [16]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng vét cạn)
cases = [
    ([1], 1),                      # n = 1: k = 1 luôn được
    ([5, 5, 5, 5], 1 + 2 + 4),      # mọi phần tử bằng nhau
    ([2, 1], 2),                   # chỉ k = 2
    ([1, 2, 3, 4, 5, 6], 1 + 2 + 3 + 6),   # đã sắp: mọi ước
    ([6, 5, 4, 3, 2, 1], 0),
    ([2, 1, 4, 3, 6, 5], 2),
    ([1, 1, 2, 1], 6),             # dễ đoán nhầm là 0 -> xem giải thích bên dưới
]
for nums, exp in cases:
    r = run_all(nums)
    print(nums, "->", r[0], "(vét cạn:", r[-1], ")")
    assert len(set(r)) == 1

[1] -> 1 (vét cạn: 1 )
[5, 5, 5, 5] -> 7 (vét cạn: 7 )
[2, 1] -> 2 (vét cạn: 2 )
[1, 2, 3, 4, 5, 6] -> 12 (vét cạn: 12 )
[6, 5, 4, 3, 2, 1] -> 0 (vét cạn: 0 )
[2, 1, 4, 3, 6, 5] -> 2 (vét cạn: 2 )
[1, 1, 2, 1] -> 6 (vét cạn: 6 )


Ca `[1, 1, 2, 1]` dễ đoán nhầm là `0` (mảng chưa sắp, có chỗ `2 > 1`). Thực ra `k = 2`: khối `[1,1]` và `[2,1]` → xoay khối sau thành `[1,2]` được `1,1,1,2` ✔; `k = 4`: xoay trái 3 bước được `[1, 1, 1, 2]` ✔. Đáp án `2 + 4 = 6`. Đây là lý do luôn kiểm chứng ca biên bằng vét cạn.

In [17]:
# 12.3 phản ví dụ cho ý tưởng sai "chỉ cần mỗi khối xoay được" (quên ranh giới)
def wrong_no_boundary(nums):
    n = len(nums); ans = 0
    for k in divisors(n):
        if all(circular_drops(nums[s:s + k]) <= 1 for s in range(0, n, k)):
            ans += k
    return ans

nums = [3, 4, 1, 2]
print("sai:", wrong_no_boundary(nums), "| đúng:", Solution().sortableIntegers(nums))

sai: 7 | đúng: 4


In [18]:
# 12.4 đối chiếu ngẫu nhiên
rng = random.Random(3886)
for _ in range(500):
    n = rng.choice([1, 2, 3, 4, 6, 8, 9, 10, 12])
    nums = [rng.randint(1, 5) for _ in range(n)]
    if rng.random() < 0.4:      # tạo mảng "gần sắp" để có nhiều k hợp lệ
        nums.sort()
        k = rng.choice(divisors(n))
        for s in range(0, n, k):
            r = rng.randrange(k); nums[s:s + k] = nums[s + r:s + k] + nums[s:s + r]
    res = run_all(nums)
    assert len(set(res)) == 1, (nums, res)
print("500 test ngẫu nhiên: OK")

500 test ngẫu nhiên: OK


In [19]:
# 12.5 test lớn đúng ràng buộc (n có nhiều ước)
rng = random.Random(5)
n = 83160
base = sorted(rng.randint(1, 10**5) for _ in range(n))
k = 7
nums = []
for s in range(0, n, k):
    b = base[s:s + k]; r = rng.randrange(k); nums += b[r:] + b[:r]
t0 = time.time(); a = Solution().sortableIntegers(nums); t1 = time.time()
b = direct_scan(nums)
print("đáp án:", a, "| khớp Cách 3:", a == b, "| thời gian < 1s:", t1 - t0 < 1)

đáp án: 7 | khớp Cách 3: True | thời gian < 1s: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên cặp vòng `(cuối, đầu)` khi đếm điểm giảm

In [20]:
def wrong_no_wrap(nums):
    n = len(nums); ans = 0
    for k in divisors(n):
        ok = True; prev_max = 0
        for s in range(0, n, k):
            b = nums[s:s + k]
            drops = sum(1 for j in range(1, k) if b[j - 1] > b[j])     # SAI: thiếu cặp vòng
            if drops > 1 or min(b) < prev_max: ok = False; break
            prev_max = max(b)
        ans += k if ok else 0
    return ans

nums = [2, 3, 1, 4]    # k = 4: chỉ 1 điểm giảm trong, nhưng có thêm điểm giảm vòng 4 > 2 -> không xoay được
print("sai:", wrong_no_wrap(nums), "| đúng:", Solution().sortableIntegers(nums))

sai: 4 | đúng: 0


### Lỗi 2 — Kiểm tra khối bằng `b == sorted(b)` (quên rằng được **xoay**)

In [21]:
def wrong_no_rotation(nums):
    n = len(nums); ans = 0
    for k in divisors(n):
        out = []; ok = True
        for s in range(0, n, k):
            b = nums[s:s + k]
            if b != sorted(b):                      # SAI: khối có thể xoay thành sắp
                ok = False; break
            out += b
        ans += k if ok and is_sorted(out) else 0
    return ans

print("sai:", wrong_no_rotation([3, 1, 2]), "| đúng:", Solution().sortableIntegers([3, 1, 2]))

sai: 0 | đúng: 3


**Ghi chú:** một lỗi "lệch một" thường gặp khác là đếm điểm giảm bằng `pref_drop[e] - pref_drop[s-1]` (tính cả cặp ranh giới). Ở bài này nó **vô hại**: nếu ranh giới có `nums[s-1] > nums[s]` thì `pref_max[s] ≥ nums[s-1] > nums[s] ≥ suff_min[s]`, khối đã bị loại bởi điều kiện ranh giới. Nhưng ở bài khác (không có điều kiện ranh giới) nó sẽ gây sai — nên viết đúng `pref_drop[e] - pref_drop[s]`.

### Lỗi 3 — Kiểm tra ranh giới bằng `nums[s-1] <= nums[s]` (giá trị **trước khi xoay**)

Sau khi xoay, phần tử cuối khối trước là **max** của khối, phần tử đầu khối sau là **min** — không phải giá trị tại vị trí gốc.

In [22]:
def wrong_raw_boundary(nums):
    n = len(nums); ans = 0
    for k in divisors(n):
        ok = all(circular_drops(nums[s:s + k]) <= 1 for s in range(0, n, k)) and \
             all(nums[s - 1] <= nums[s] for s in range(k, n, k))        # SAI
        ans += k if ok else 0
    return ans

nums = [3, 1, 2, 4]     # khối [3,1] -> [1,3]; khối [2,4]; giá trị gốc 1 <= 2 nhưng max 3 > min 2
print("sai:", wrong_raw_boundary(nums), "| đúng:", Solution().sortableIntegers(nums))

sai: 2 | đúng: 0


### Lỗi 4 — Duyệt mọi `k` từ 1 tới `n` (không chỉ ước)

Ngoài việc chậm, `k` không chia hết `n` không hợp lệ theo đề. Luôn lọc `n % k == 0`.

In [23]:
nums = [1, 2, 3, 4, 5, 6]
print("cộng cả k không phải ước (sai):", sum(range(1, 7)), "| đúng:", Solution().sortableIntegers(nums))

cộng cả k không phải ước (sai): 21 | đúng: 12


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Xoay vòng để được dãy sắp"** ⇒ đếm **điểm giảm vòng ≤ 1** (LC 1752 *Check if Array Is Sorted and Rotated*).
- **"Chia thành các khối độ dài `k`, `k | n`"** ⇒ duyệt ước; tổng số khối trên mọi ước là `σ(n)` — nhỏ.
- **"Mọi phần tử bên trái ≤ mọi phần tử bên phải"** tại vị trí cắt ⇒ `pref_max[i] ≤ suff_min[i]` (LC 915 *Partition Array into Disjoint Intervals*, LC 769 *Max Chunks To Make Sorted*).
- **"`B` có phải xoay của `A`"** tổng quát ⇒ tìm `B` trong `A + A` (KMP / Z-function).

## 15. Biến thể của bài

1. **Cho phép đảo ngược khối** thay vì xoay: khối phải không giảm hoặc không tăng.
2. **Khối không cần bằng nhau** (chọn cách chia tối đa số khối): *Max Chunks To Make Sorted II* (LC 768) với phép **sắp xếp** khối.
3. **Xoay toàn bộ mảng** chỉ một lần: LC 1752.
4. **Đếm số `k`** thay vì tổng; hoặc tìm `k` nhỏ nhất sắp được.

In [24]:
# Biến thể 1: mỗi khối được ĐẢO NGƯỢC (hoặc giữ nguyên) thay vì xoay
def reverse_variant(nums):
    n = len(nums); ans = 0
    for k in divisors(n):
        out = []; ok = True
        for s in range(0, n, k):
            b = nums[s:s + k]
            if is_sorted(b): out += b
            elif is_sorted(b[::-1]): out += b[::-1]
            else: ok = False; break
        if ok and is_sorted(out): ans += k
    return ans

print(reverse_variant([2, 1, 4, 3]), reverse_variant([3, 2, 1, 6, 5, 4]), reverse_variant([1, 3, 2, 4]))

2 3 0


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi phép xoay | Mô phỏng | `O(n σ(n))` | `O(n)` | Kiểm thử |
| 2. KMP | `sorted(b)` trong `b + b` | `O(n log n d(n))` | `O(n)` | Đích tổng quát |
| 3. Đếm điểm giảm | `O(k)` mỗi khối | `O(n d(n))` | `O(1)` | Dễ viết, đủ nhanh |
| 4. Tiền tố | `O(1)` mỗi khối | `O(n + σ(n))` | `O(n)` | **Lời giải chuẩn** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `nums = [4, 1, 2, 3]` → ?

**Bài 2 (tính tay).** `nums = [2, 3, 1, 5, 6, 4]` — lập bảng cho mọi ước.

**Bài 3 (điền TODO).** Viết `can_rotate_sorted(b)` trả về `True` nếu khối xoay được thành không giảm.

**Bài 4 (tự viết lại).** Viết lại lời giải chính nhưng thay `pref_max/suff_min` bằng mảng `good_cut[i] = pref_max[i] <= suff_min[i]`.

**Bài 5 (mini challenge).** Tìm **`k` nhỏ nhất** sắp được (hoặc `-1`), và với `k` đó, in ra số bước xoay trái của từng khối.

In [25]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def can_rotate_sorted(b):
    # TODO: đếm j với b[j] > b[(j+1) % len(b)], trả về số đó <= 1
    pass

In [26]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def smallest_k_and_shifts(nums):
    # TODO: duyệt ước tăng dần; với khối có điểm giảm tại j thì xoay trái (j+1) bước
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Ước `1, 2, 4`. (2) `k = 4`: một điểm giảm `4 > 1`, cặp vòng `3 > 4`? không ⇒ xoay được. (3) `k = 1, 2` hỏng ⇒ đáp án `4`.

**Bài 2.** (1) `k = 3`: khối `[2,3,1]` và `[5,6,4]`, mỗi khối 1 điểm giảm. (2) Ranh giới: `max = 3 ≤ min = 4` ✔. (3) `k = 6`: điểm giảm `3>1`, `6>4`, vòng `4>2` ⇒ 3 ✘; `k = 1, 2` ✘ ⇒ đáp án `3`.

**Bài 3.** (1) Vòng lặp `j` từ `0` tới `k-1`. (2) Chỉ số `(j + 1) % k`. (3) `k = 1` luôn `True`.

**Bài 4.** (1) Tính `good_cut` một lần. (2) Trong `ok(k)` dùng `good_cut[s]`. (3) So kết quả.

**Bài 5.** (1) Ước tăng dần, dừng ở cái đầu tiên. (2) Nếu khối không có điểm giảm trong thì 0 bước (hoặc phân biệt toàn bằng nhau). (3) Kiểm tra lại bằng cách xoay thật.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1, 2
assert Solution().sortableIntegers([4, 1, 2, 3]) == 4 == brute([4, 1, 2, 3])
assert Solution().sortableIntegers([2, 3, 1, 5, 6, 4]) == 3 == brute([2, 3, 1, 5, 6, 4])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [28]:
# Bài 3
def can_rotate_sorted_sol(b):
    k = len(b)
    return sum(1 for j in range(k) if b[j] > b[(j + 1) % k]) <= 1

rng = random.Random(3)
for _ in range(500):
    b = [rng.randint(1, 4) for _ in range(rng.randint(1, 6))]
    assert can_rotate_sorted_sol(b) == any(is_sorted(b[r:] + b[:r]) for r in range(len(b)))
print("Bài 3 OK")

Bài 3 OK


In [29]:
# Bài 4
def sol_good_cut(nums):
    n = len(nums)
    pm = [0] * (n + 1); sm = [10**9] * (n + 1); pd = [0] * n
    for i, x in enumerate(nums): pm[i + 1] = max(pm[i], x)
    for i in range(n - 1, -1, -1): sm[i] = min(nums[i], sm[i + 1])
    for i in range(1, n): pd[i] = pd[i - 1] + (nums[i - 1] > nums[i])
    good_cut = [pm[i] <= sm[i] for i in range(n + 1)]
    def ok(k):
        return all(good_cut[s] and pd[s + k - 1] - pd[s] + (nums[s + k - 1] > nums[s]) <= 1 for s in range(0, n, k))
    return sum(k for k in divisors(n) if ok(k))

rng = random.Random(4)
for _ in range(300):
    n = rng.choice([1, 2, 4, 6, 8, 12]); nums = [rng.randint(1, 6) for _ in range(n)]
    assert sol_good_cut(nums) == Solution().sortableIntegers(nums)
print("Bài 4 OK")

Bài 4 OK


In [30]:
# Bài 5
def smallest_k_and_shifts_sol(nums):
    n = len(nums)
    for k in divisors(n):
        shifts = []; out = []; ok = True
        for s in range(0, n, k):
            b = nums[s:s + k]
            r = next((r for r in range(k) if is_sorted(b[r:] + b[:r])), None)
            if r is None: ok = False; break
            shifts.append(r); out += b[r:] + b[:r]
        if ok and is_sorted(out):
            return k, shifts
    return -1, []

print(smallest_k_and_shifts_sol([2, 1, 4, 3, 6, 5]), smallest_k_and_shifts_sol([2, 3, 1, 5, 6, 4]), smallest_k_and_shifts_sol([7, 6, 5]))
assert smallest_k_and_shifts_sol([2, 3, 1, 5, 6, 4]) == (3, [2, 2])
print("Bài 5 OK")

(2, [1, 1, 1]) (3, [2, 2]) (-1, [])
Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
pref_max[i+1] = max(pref_max[i], nums[i]); suff_min[i] = min(nums[i], suff_min[i+1])
pref_drop[i] = pref_drop[i-1] + (nums[i-1] > nums[i])
def ok(k):
    for s in range(0, n, k):
        e = s + k - 1
        drops = pref_drop[e] - pref_drop[s] + (nums[e] > nums[s])
        if drops > 1 or pref_max[s] > suff_min[s]: return False
    return True
ans = sum(k for k in ước(n) if ok(k))
```

**Checklist:**

- [ ] Chỉ duyệt **ước** của `n` (cặp `i`, `n/i`, không cộng trùng khi `i = n/i`)?
- [ ] Điểm giảm trong khối dùng `pref_drop[e] - pref_drop[s]` (không gồm cặp ranh giới)?
- [ ] Cộng cặp vòng `nums[e] > nums[s]`?
- [ ] Ranh giới dùng `pref_max[s] ≤ suff_min[s]` (giá trị sau xoay), không dùng giá trị gốc kề nhau?
- [ ] `k = 1` và `k = n` được xử lý đúng?